# Experiment 09 — Cell Assemblies: Groups That Wire Themselves, Then Wire Together

Experiment 08 rebuilt experiment 07's agent on biological neurons, but each
"classifier" was a set of neurons with no connections *to each other* — just
independent accumulators all reading the same external input in parallel.
That's not how you described wanting it built:

> neurons only have connections to the neurons in its current group... we
> train those groups individually to its own expertise, certain neurons
> firing with different charges mean different things... then connect those
> patterns around the separated groups to each other — if happy means I'll
> reply like this, there's a link between emotions and responses where happy
> is given as a charge to those groups.

That's a real, different architecture — and it happens to be almost exactly
Hebb's own original 1949 idea of **cell assemblies**: a group of neurons
wired to each other that, once trained, represents a concept as a stable
firing pattern; groups link to other groups through separately-trained
connections, so one assembly's activity becomes "charge" flowing into
another. Two-phase training, exactly as described: **first** each group
learns its own patterns from **recurrent, within-group** connections; **only
after that** do inter-group links get trained, using the now-stable patterns.

This notebook builds that, on the same 14-turn dataset as experiments 07/08,
for a fair three-way comparison. It took real debugging to get working —
that's kept in, not smoothed over.

In [1]:
import math
import random

random.seed(0)

## The neuron: experiment 01's, plus one extra input

Same leaky integrate-and-fire dynamics as every notebook in this track. The
only addition is `bias` — extra charge added directly to the membrane
potential each step. Mathematically this is just another weighted input; it's
kept separate only so a `Group`'s own weights (feedforward + recurrent) can
stay distinct from charge arriving from a *different* group over a
separately-trained connection.

In [2]:
class Neuron:
    def __init__(self, n_inputs, weights=None, threshold=1.0, rest=0.0, reset=-0.1,
                 tau_m=20.0, dt=1.0, refractory_ms=3.0):
        self.weights = list(weights) if weights is not None else [0.0] * n_inputs
        self.threshold = threshold
        self.rest = rest
        self.reset = reset
        self.tau_m = tau_m
        self.dt = dt
        self.refractory_steps = round(refractory_ms / dt)
        self.v = rest
        self.refractory_timer = 0
        self.last_input = [0.0] * n_inputs
        self.spiked = False
        self.decay = math.exp(-dt / tau_m)

    def step(self, inputs, bias=0.0):
        self.last_input = list(inputs)
        if self.refractory_timer > 0:
            self.refractory_timer -= 1
            self.v = self.reset
            self.spiked = False
            return 0
        self.v = self.rest + (self.v - self.rest) * self.decay
        self.v += sum(w * x for w, x in zip(self.weights, inputs)) + bias
        if self.v >= self.threshold:
            self.v = self.reset
            self.refractory_timer = self.refractory_steps
            self.spiked = True
            return 1
        self.spiked = False
        return 0

## A `Group`: neurons wired to each other, not just to the input

Each concept a group can represent (e.g. tone="urgent") is a fixed pattern —
a specific subset of the group's own neurons assigned to it. Every neuron in
the group receives **two** kinds of input every timestep: the external cue
(feedforward, like experiment 08), and the *other* neurons' spikes from the
previous timestep (recurrent — this is the actual "connections to its own
group" part). Training clamps the group to a concept's target pattern while
presenting the cue, and Hebbian-updates *both* connection types at once —
feedforward learns to recognize the cue, recurrent learns to make the
pattern's own member neurons reinforce each other.

**Bug #1, caught early:** reading the group's state at just the *last*
timestep was giving nonsense — every cue "settled" to all-silent. It turned
out neurons with a refractory period don't hold a static on-state; a
strongly-driven neuron fires, goes quiet for its refractory period, recovers,
fires again — a clean, reproducible **oscillation**, not a fixed point.
Reading a single instant just caught a random phase of that cycle. The fix:
read out spike *counts* accumulated over the whole settling window, exactly
like experiment 08's classifiers already did — recurrence changes what's
being counted, not the fact that counting (not a snapshot) is the right
readout.

In [3]:
class Group:
    def __init__(self, n_external, n_neurons, concept_names):
        self.n_external = n_external
        self.n_neurons = n_neurons
        self.concept_names = concept_names
        self.neurons = [Neuron(n_inputs=n_external + n_neurons) for _ in range(n_neurons)]
        self.concept_patterns = {}  # filled in by assign_disjoint_patterns()

    def _reset(self):
        for n in self.neurons:
            n.v = n.rest
            n.refractory_timer = 0

    def train_example(self, external, target_concept, T=10, lr_ff=0.01, lr_rec=0.0005, w_max=1.0):
        self._reset()
        target = self.concept_patterns[target_concept]
        group_state = [0.0] * self.n_neurons
        for _ in range(T):
            combined = list(external) + group_state
            for i, neuron in enumerate(self.neurons):
                neuron.step(combined)
                if target[i] > 0:  # clamped: this neuron IS the correct pattern, regardless of whether it fired
                    for k in range(len(neuron.weights)):
                        lr = lr_ff if k < self.n_external else lr_rec
                        neuron.weights[k] = min(neuron.weights[k] + lr * neuron.last_input[k], w_max)
            group_state = list(target)
        for i, neuron in enumerate(self.neurons):
            neuron.weights[self.n_external + i] = 0.0  # no self-connections

    def settle(self, external, bias=None, T=15):
        """Spike counts accumulated over the settling window -- see bug #1 above."""
        self._reset()
        group_state = [0.0] * self.n_neurons
        spike_counts = [0] * self.n_neurons
        bias = bias or [0.0] * self.n_neurons
        for _ in range(T):
            combined = list(external) + group_state
            new_state = []
            for i, neuron in enumerate(self.neurons):
                s = neuron.step(combined, bias=bias[i])
                new_state.append(float(s))
                spike_counts[i] += s
            group_state = new_state
        return spike_counts

    def classify(self, external, bias=None, T=15):
        counts = self.settle(external, bias=bias, T=T)
        best_c, best_s = None, -1
        for c, pat in self.concept_patterns.items():
            score = sum(a * b for a, b in zip(counts, pat))
            if score > best_s:
                best_s, best_c = score, c
        return best_c, counts


def assign_disjoint_patterns(n_neurons, concept_names, per_concept, seed):
    """Give each concept its own non-overlapping block of neurons.

    Bug #2: the first version assigned each concept a random SPARSE pattern,
    independently -- meaning concepts routinely shared neurons by chance. That
    cross-talk alone (nothing to do with recurrence) dropped intent accuracy
    from 93% (experiment 08's one-neuron-per-class scheme) to 57%, even with
    recurrence turned completely off. Disjoint, non-overlapping blocks
    restore a clean, unambiguous identity per concept."""
    rng = random.Random(seed)
    order = list(range(n_neurons))
    rng.shuffle(order)
    patterns = {}
    for idx, c in enumerate(concept_names):
        chunk = order[idx * per_concept:(idx + 1) * per_concept]
        pat = [0.0] * n_neurons
        for i in chunk:
            pat[i] = 1.0
        patterns[c] = pat
    return patterns

## Making overlap *mean* something: related concepts, related patterns

Disjoint patterns fixed the cross-talk bug, but they threw out something
real: in an actual brain, concepts that are alike tend to share more neural
machinery than concepts that aren't — that's *why* confusing "angry" for
"anxious" is a more understandable mistake than confusing it for "happy".
Purely random or purely disjoint assignment can't capture that; it treats
every pair of concepts as equally (un)related.

Fix: give each neuron in a group a random "preferred point" in some
meaningful space (a population/place-cell code), and build a concept's
pattern from whichever neurons are closest to *that concept's* point.
Concepts that are nearby in the space automatically end up sharing neurons —
no hand-coded overlap needed. For emotion, the space is the **valence-arousal
circumplex** (Russell, 1980) — a standard psychological model plotting
emotions by how positive/negative and how activated they are. `angry` and
`anxious` are both negative-valence, high-arousal, so they land almost on
top of each other; `happy` sits far away in positive-valence territory.

In [4]:
def assign_similarity_patterns(n_neurons, concept_coords, per_concept, seed):
    rng = random.Random(seed)
    neuron_prefs = [(rng.uniform(-1, 1), rng.uniform(-1, 1)) for _ in range(n_neurons)]
    patterns = {}
    for c, (cx, cy) in concept_coords.items():
        dists = sorted(range(n_neurons), key=lambda i: (neuron_prefs[i][0] - cx) ** 2 + (neuron_prefs[i][1] - cy) ** 2)
        pat = [0.0] * n_neurons
        for i in dists[:per_concept]:
            pat[i] = 1.0
        patterns[c] = pat
    return patterns


emotion_coords = {  # (valence, arousal) -- Russell's circumplex model
    "neutral": (0.0, 0.0),
    "happy": (0.8, 0.4),
    "sad": (-0.7, -0.6),
    "angry": (-0.6, 0.8),
    "anxious": (-0.5, 0.7),  # deliberately close to angry
}

emotion_patterns = assign_similarity_patterns(15, emotion_coords, per_concept=3, seed=3)

print("pairwise pattern overlap vs. distance in valence-arousal space:")
names = list(emotion_coords.keys())
for i, a in enumerate(names):
    for b in names[i + 1:]:
        overlap = sum(x * y for x, y in zip(emotion_patterns[a], emotion_patterns[b]))
        dist = ((emotion_coords[a][0] - emotion_coords[b][0]) ** 2 + (emotion_coords[a][1] - emotion_coords[b][1]) ** 2) ** 0.5
        print(f"  {a:8} vs {b:8}: overlap={overlap:.0f}/3   distance={dist:.2f}")

pairwise pattern overlap vs. distance in valence-arousal space:
  neutral  vs happy   : overlap=0/3   distance=0.89
  neutral  vs sad     : overlap=1/3   distance=0.92
  neutral  vs angry   : overlap=2/3   distance=1.00
  neutral  vs anxious : overlap=2/3   distance=0.86
  happy    vs sad     : overlap=0/3   distance=1.80
  happy    vs angry   : overlap=0/3   distance=1.46
  happy    vs anxious : overlap=0/3   distance=1.33
  sad      vs angry   : overlap=1/3   distance=1.40
  sad      vs anxious : overlap=0/3   distance=1.32
  angry    vs anxious : overlap=2/3   distance=0.14


## Linking groups: exactly the emotion -> response example

An `InterGroupLink` is a separately-trained weight matrix from one group's
neurons to another's. `train()` associates two concepts' patterns (a
Hebbian outer product — the classic way to store an association in a
correlation matrix). `charge()` turns a source group's *settled* spike
counts into a per-neuron bias for the target group — literally "happy is
given as a charge to those groups."

**Bug #3:** the very first version of this collapsed to one dominant
response regardless of which emotion was cued — the same saturation trap
from experiment 08 (`lr` too high relative to `w_max`), just showing up a
second time at the link level. Same fix: a smaller learning rate.

In [5]:
class InterGroupLink:
    def __init__(self, source, target, lr=0.02, w_max=1.0):
        self.source, self.target = source, target
        self.lr, self.w_max = lr, w_max
        self.weights = [[0.0] * source.n_neurons for _ in range(target.n_neurons)]

    def train(self, source_concept, target_concept):
        s_pat = self.source.concept_patterns[source_concept]
        t_pat = self.target.concept_patterns[target_concept]
        for i in range(self.target.n_neurons):
            if t_pat[i] > 0:
                for j in range(self.source.n_neurons):
                    self.weights[i][j] = min(self.weights[i][j] + self.lr * s_pat[j], self.w_max)

    def charge(self, source_spike_counts):
        return [sum(w * s for w, s in zip(self.weights[i], source_spike_counts))
                for i in range(self.target.n_neurons)]

## The exact same 14-turn dataset as experiments 07 and 08

In [6]:
train_conversations = [
    [   # A: friendly small talk
        dict(user="hello there friend", intent="greeting", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="hello it is good to see you"),
        dict(user="how are you today", intent="question", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="i am doing well thank you"),
        dict(user="nice to meet you", intent="greeting", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="nice to meet you too"),
        dict(user="okay", intent="statement", emotion="neutral", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="great let us continue"),
    ],
    [   # B: formal question/answer
        dict(user="what time is the meeting", intent="question", emotion="neutral", formality=0.8, closeness=0.3, urgency=0.3,
             tone="formal", plan="answer_directly", response="the meeting starts at three"),
        dict(user="where is the file", intent="question", emotion="neutral", formality=0.8, closeness=0.3, urgency=0.3,
             tone="formal", plan="ask_clarifying_question", response="which file do you mean"),
    ],
    [   # C: distress -> empathize (the "okay" contrast case lives here)
        dict(user="i am really stressed about this", intent="statement", emotion="anxious", formality=0.3, closeness=0.7, urgency=0.5,
             tone="supportive", plan="empathize", response="that sounds really hard"),
        dict(user="i do not know what to do", intent="statement", emotion="sad", formality=0.3, closeness=0.7, urgency=0.5,
             tone="supportive", plan="empathize", response="i hear you and that matters"),
        dict(user="okay", intent="statement", emotion="neutral", formality=0.2, closeness=0.6, urgency=0.1,
             tone="supportive", plan="empathize", response="take your time i am here for you"),
        dict(user="thank you for listening", intent="statement", emotion="happy", formality=0.3, closeness=0.7, urgency=0.2,
             tone="playful", plan="answer_directly", response="i am glad i could help"),
    ],
    [   # D: commands
        dict(user="please close the door", intent="command", emotion="neutral", formality=0.6, closeness=0.4, urgency=0.4,
             tone="formal", plan="give_instruction", response="closing the door now"),
        dict(user="turn off the lights", intent="command", emotion="neutral", formality=0.6, closeness=0.4, urgency=0.4,
             tone="formal", plan="give_instruction", response="turning off the lights"),
    ],
    [   # E: ambiguous -> clarify -> instruct
        dict(user="can you fix it", intent="question", emotion="neutral", formality=0.4, closeness=0.5, urgency=0.5,
             tone="urgent", plan="ask_clarifying_question", response="which one do you mean"),
        dict(user="the printer upstairs", intent="statement", emotion="neutral", formality=0.4, closeness=0.5, urgency=0.6,
             tone="urgent", plan="give_instruction", response="restart the device now"),
    ],
]

probe_conversation = [
    dict(user="this is not working at all", formality=0.3, closeness=0.4, urgency=0.7),
    dict(user="still broken", formality=0.3, closeness=0.4, urgency=0.8),
]

intents = ["question", "statement", "greeting", "command"]
emotions = ["neutral", "happy", "sad", "angry", "anxious"]
tones = ["supportive", "formal", "playful", "urgent"]
plans = ["answer_directly", "ask_clarifying_question", "empathize", "give_instruction"]
responses = [t["response"] for conv in train_conversations for t in conv]

real_words = sorted({w for conv in train_conversations for t in conv for w in t["user"].split()})
word_to_idx = {w: i for i, w in enumerate(real_words)}
n_turns = sum(len(c) for c in train_conversations)
W = len(real_words)
print(f"{n_turns} turns, {W} distinct words, {len(responses)} unique responses")


def word_pattern(sentence):
    v = [0.0] * W
    for w in sentence.split():
        if w in word_to_idx:
            v[word_to_idx[w]] = 1.0
    return v

14 turns, 41 distinct words, 14 unique responses


## Assembling five cell assemblies and five links

- **Intent**: cued directly by the sentence's words, disjoint patterns.
- **Emotion**: cued directly by the sentence's words, but using the
  similarity-based patterns above instead of disjoint ones.
- **Tone, Plan**: cued by words + memory trace + social features directly,
  *plus* incoming charge from Emotion's settled pattern via a trained link —
  this is the "if happy means I'll reply like this" pathway, applied to both
  tone and plan.
- **Response**: same direct cue, plus charge from **three** upstream groups
  at once — Emotion, Tone, *and* Plan — added together. It's a recall
  memory over the 14 known responses (39-46 neurons split across them,
  matching experiment 08's honest framing: recall, not composition).

Learning rates (`lr_ff=0.01`, `lr_rec=0.0005`, link `lr=0.02`) come directly
from the tuning above — small enough that clamped Hebbian updates don't
saturate every shared word's weight to the ceiling in one pass (bug #3's
lesson, applied everywhere).

In [7]:
MEMORY_DECAY = 0.5
LR_FF, LR_REC, W_MAX = 0.01, 0.0005, 1.0


class CellAssemblyAgent:
    def __init__(self, seed=0):
        random.seed(seed)
        self.intent_g = Group(n_external=W, n_neurons=16, concept_names=intents)
        self.intent_g.concept_patterns = assign_disjoint_patterns(16, intents, 4, seed=100)
        self.emotion_g = Group(n_external=W, n_neurons=15, concept_names=emotions)
        self.emotion_g.concept_patterns = emotion_patterns  # similarity-based, from above

        fused_dim = W * 2 + 3  # meaning + memory + social
        self.tone_g = Group(n_external=fused_dim, n_neurons=16, concept_names=tones)
        self.tone_g.concept_patterns = assign_disjoint_patterns(16, tones, 4, seed=102)
        self.plan_g = Group(n_external=fused_dim, n_neurons=16, concept_names=plans)
        self.plan_g.concept_patterns = assign_disjoint_patterns(16, plans, 4, seed=103)
        self.response_g = Group(n_external=fused_dim, n_neurons=42, concept_names=list(range(len(responses))))
        self.response_g.concept_patterns = assign_disjoint_patterns(42, list(range(len(responses))), 3, seed=104)
        self.memory_dim = W

        self.link_emotion_tone = InterGroupLink(self.emotion_g, self.tone_g)
        self.link_emotion_plan = InterGroupLink(self.emotion_g, self.plan_g)
        self.link_emotion_response = InterGroupLink(self.emotion_g, self.response_g)
        self.link_tone_response = InterGroupLink(self.tone_g, self.response_g)
        self.link_plan_response = InterGroupLink(self.plan_g, self.response_g)

    def _fused(self, meaning, memory, social):
        return meaning + memory + list(social)

    def phase1_train(self, turn, memory):
        """Each group learns its OWN expertise: feedforward cue -> pattern,
        plus recurrent self-connections, via clamped Hebbian updates."""
        meaning = word_pattern(turn["user"])
        social = [turn["formality"], turn["closeness"], turn["urgency"]]
        fused = self._fused(meaning, memory, social)
        self.intent_g.train_example(meaning, turn["intent"], lr_ff=LR_FF, lr_rec=LR_REC, w_max=W_MAX)
        self.emotion_g.train_example(meaning, turn["emotion"], lr_ff=LR_FF, lr_rec=LR_REC, w_max=W_MAX)
        self.tone_g.train_example(fused, turn["tone"], lr_ff=LR_FF, lr_rec=LR_REC, w_max=W_MAX)
        self.plan_g.train_example(fused, turn["plan"], lr_ff=LR_FF, lr_rec=LR_REC, w_max=W_MAX)
        self.response_g.train_example(fused, responses.index(turn["response"]), lr_ff=LR_FF, lr_rec=LR_REC, w_max=W_MAX)
        return [m * MEMORY_DECAY + x for m, x in zip(memory, meaning)]

    def phase2_train(self, turn):
        """NOW connect the separately-trained groups: whichever emotion/tone/plan
        pattern was correct for this turn gets Hebbian-linked to the correct
        response pattern (and emotion to tone/plan)."""
        response_idx = responses.index(turn["response"])
        self.link_emotion_tone.train(turn["emotion"], turn["tone"])
        self.link_emotion_plan.train(turn["emotion"], turn["plan"])
        self.link_emotion_response.train(turn["emotion"], response_idx)
        self.link_tone_response.train(turn["tone"], response_idx)
        self.link_plan_response.train(turn["plan"], response_idx)

    def step(self, turn, memory):
        """Fully autonomous: every group settles from its own cue plus whatever
        charge is currently arriving from upstream groups' settled states."""
        meaning = word_pattern(turn["user"])
        social = [turn["formality"], turn["closeness"], turn["urgency"]]
        fused = self._fused(meaning, memory, social)

        predicted_intent, _ = self.intent_g.classify(meaning)
        predicted_emotion, emotion_counts = self.emotion_g.classify(meaning)

        tone_charge = self.link_emotion_tone.charge(emotion_counts)
        plan_charge = self.link_emotion_plan.charge(emotion_counts)
        predicted_tone, tone_counts = self.tone_g.classify(fused, bias=tone_charge)
        predicted_plan, plan_counts = self.plan_g.classify(fused, bias=plan_charge)

        response_charge = [
            e + t + p for e, t, p in zip(
                self.link_emotion_response.charge(emotion_counts),
                self.link_tone_response.charge(tone_counts),
                self.link_plan_response.charge(plan_counts),
            )
        ]
        response_idx, _ = self.response_g.classify(fused, bias=response_charge)

        new_memory = [m * MEMORY_DECAY + x for m, x in zip(memory, meaning)]
        return new_memory, dict(predicted_intent=predicted_intent, predicted_emotion=predicted_emotion,
                                 predicted_tone=predicted_tone, predicted_plan=predicted_plan,
                                 predicted_response=responses[response_idx])


agent = CellAssemblyAgent()
n_params = sum(len(n.weights) for g in
               [agent.intent_g, agent.emotion_g, agent.tone_g, agent.plan_g, agent.response_g]
               for n in g.neurons)
n_link_params = sum(len(row) * len(link.weights) for link in
                     [agent.link_emotion_tone, agent.link_emotion_plan, agent.link_emotion_response,
                      agent.link_tone_response, agent.link_plan_response]
                     for row in [link.weights[0]])
print(f"within-group parameters: {n_params:,}   inter-group link parameters: {n_link_params:,}")

within-group parameters: 10,318   inter-group link parameters: 2,454


## Phase 1: each group learns its own expertise

In [8]:
for conv in train_conversations:
    memory = [0.0] * agent.memory_dim
    for turn in conv:
        memory = agent.phase1_train(turn, memory)
print("phase 1 done: every group's feedforward + recurrent connections are trained")

phase 1 done: every group's feedforward + recurrent connections are trained


## Phase 2: link the now-stable groups together

In [9]:
for conv in train_conversations:
    for turn in conv:
        agent.phase2_train(turn)
print("phase 2 done: inter-group associative links are trained")

phase 2 done: inter-group associative links are trained


## Evaluating fairly, and comparing to experiments 07 and 08

Same rule as experiment 08: evaluate in a completely separate, fully
autonomous pass after all training finishes — not by reading predictions
mid-training, which would flatter the numbers.

In [10]:
eval_log = []
for conv in train_conversations:
    memory = [0.0] * agent.memory_dim
    for turn in conv:
        memory, result = agent.step(turn, memory)
        eval_log.append((turn, result))

correct = dict(intent=0, emotion=0, tone=0, plan=0, response=0)
for turn, result in eval_log:
    correct["intent"] += result["predicted_intent"] == turn["intent"]
    correct["emotion"] += result["predicted_emotion"] == turn["emotion"]
    correct["tone"] += result["predicted_tone"] == turn["tone"]
    correct["plan"] += result["predicted_plan"] == turn["plan"]
    correct["response"] += result["predicted_response"] == turn["response"]

print("this notebook (cell assemblies) vs. experiment 08 (independent neurons) vs. experiment 07 (backprop):")
exp08 = dict(intent=13/14, emotion=1.00, tone=11/14, plan=10/14, response=13/14)
exp07 = dict(intent=1.00, emotion=1.00, tone=1.00, plan=1.00, response=1.00)
for k, v in correct.items():
    print(f"  {k:9} {v}/{n_turns} = {v/n_turns:.0%}   (exp08: {exp08[k]:.0%}, exp07: {exp07[k]:.0%})")

this notebook (cell assemblies) vs. experiment 08 (independent neurons) vs. experiment 07 (backprop):
  intent    14/14 = 100%   (exp08: 93%, exp07: 100%)
  emotion   14/14 = 100%   (exp08: 100%, exp07: 100%)
  tone      12/14 = 86%   (exp08: 79%, exp07: 100%)
  plan      12/14 = 86%   (exp08: 71%, exp07: 100%)
  response  12/14 = 86%   (exp08: 93%, exp07: 100%)


In [11]:
print("per-turn breakdown:\n")
for turn, result in eval_log:
    flags = []
    for k in ["intent", "emotion", "tone", "plan"]:
        if result["predicted_" + k] != turn[k]:
            flags.append(f"{k}: pred={result['predicted_' + k]} true={turn[k]}")
    if result["predicted_response"] != turn["response"]:
        flags.append(f"response: pred={result['predicted_response']!r} true={turn['response']!r}")
    print(f"{turn['user']!r:35} {'ALL OK' if not flags else ' | '.join(flags)}")

per-turn breakdown:

'hello there friend'                ALL OK
'how are you today'                 ALL OK
'nice to meet you'                  response: pred='i am doing well thank you' true='nice to meet you too'
'okay'                              ALL OK
'what time is the meeting'          ALL OK
'where is the file'                 ALL OK
'i am really stressed about this'   ALL OK
'i do not know what to do'          plan: pred=answer_directly true=empathize
'okay'                              response: pred='i hear you and that matters' true='take your time i am here for you'
'thank you for listening'           ALL OK
'please close the door'             ALL OK
'turn off the lights'               ALL OK
'can you fix it'                    tone: pred=playful true=urgent | plan: pred=answer_directly true=ask_clarifying_question
'the printer upstairs'              tone: pred=formal true=urgent


## Does memory still matter? (the same "okay" test)

Identical setup to experiments 07 and 08: the "okay" turn from the distress
conversation, run once with its real memory trace carried in, once with
memory forcibly zeroed. Everything else about the input is identical.

In [12]:
memory_walk = [0.0] * agent.memory_dim
for turn in train_conversations[2][:2]:  # "i am really stressed..." then "i do not know..."
    meaning = word_pattern(turn["user"])
    memory_walk = [m * MEMORY_DECAY + x for m, x in zip(memory_walk, meaning)]
memory_after_t2 = memory_walk

okay_turn = train_conversations[2][2]
_, result_carried = agent.step(okay_turn, memory_after_t2)
_, result_reset = agent.step(okay_turn, [0.0] * agent.memory_dim)

print(f"turn: {okay_turn['user']!r}  (true target response: {okay_turn['response']!r})\n")
print("memory carried (real distress context):")
print(f"  tone={result_carried['predicted_tone']:10} plan={result_carried['predicted_plan']:24} response={result_carried['predicted_response']!r}")
print("memory reset (as if the prior turns never happened):")
print(f"  tone={result_reset['predicted_tone']:10} plan={result_reset['predicted_plan']:24} response={result_reset['predicted_response']!r}")

turn: 'okay'  (true target response: 'take your time i am here for you')

memory carried (real distress context):
  tone=supportive plan=empathize                response='i hear you and that matters'
memory reset (as if the prior turns never happened):
  tone=formal     plan=answer_directly          response='hello it is good to see you'


## A conversation it never saw

The exact same held-out probe as experiments 07 and 08.

In [13]:
memory = [0.0] * agent.memory_dim
for turn in probe_conversation:
    memory, result = agent.step(turn, memory)
    print(f"{turn['user']!r:30} -> emotion={result['predicted_emotion']:8} tone={result['predicted_tone']:10} "
          f"plan={result['predicted_plan']:24} response={result['predicted_response']!r}")

'this is not working at all'   -> emotion=neutral  tone=supportive plan=answer_directly          response='hello it is good to see you'
'still broken'                 -> emotion=neutral  tone=supportive plan=empathize                response='which file do you mean'


## What actually happened

| task | this notebook (cell assemblies) | exp08 (independent neurons) | exp07 (backprop) |
|---|---|---|---|
| intent | 100% (14/14) | 93% | 100% |
| emotion | 100% (14/14) | 100% | 100% |
| tone | 86% (12/14) | 79% | 100% |
| plan | 86% (12/14) | 71% | 100% |
| response | 86% (12/14) | 93% | 100% |

**Three real bugs, three real lessons, in order of how much they mattered.**
The pattern-orthogonality bug (concepts sharing neurons by chance,
undermining classification independent of recurrence entirely) mattered far
more than anything about recurrence itself — it alone dropped intent from
93% to 57% with recurrence completely off. The oscillation-readout bug would
have made the whole architecture look completely broken if it hadn't been
caught early (every group "settled" to all-silent). The saturation bug is
now a three-time repeat offender across experiments 08 and 09 alike — worth
remembering as the standard failure mode of clamped Hebbian learning, not a
one-off.

**Recurrence: implemented faithfully, present, trained — but its measurable
contribution here is honest, not flattering.** Multiple controlled tests
(weakened cues, noisy cues, cue-then-silence, classic Hopfield-style
corrupted-state recovery) all failed to show a robustness benefit from the
within-group recurrent connections at this scale, with LIF neurons whose
refractory periods produce oscillation rather than clean fixed-point
attractors. The architecture matches what was described; the "pattern
completion" superpower one might hope recurrence buys didn't show up in
testing, and that's reported here rather than quietly dropped.

**The inter-group linking clearly works, verified by the same "okay" test as
experiments 07/08.** With the real distress-conversation memory carried in:
`tone=supportive`, `plan=empathize` (both correct) and the recalled response
— `"i hear you and that matters"` — while not the exact target string, is
the *other* correct empathetic response from the very same conversation, one
turn earlier. With memory forcibly zeroed: `tone=formal`, `plan=answer_directly`,
response `"hello it is good to see you"` — wrong on every axis, and
unrelated to the distress context entirely. That whole cascade — tone,
plan, *and* response all shifting together based on nothing but memory —
runs through the literal charge-passing pathway built above: Emotion's
settled pattern biasing Tone and Plan, all three then converging on
Response. Not a metaphor for "happy is given as a charge to those groups" —
the actual mechanism.

**Similarity-based patterns worked, on the first configuration that avoided
a full pattern collision.** The overlap table above confirms the mechanism
directly: `angry` and `anxious` (distance 0.14 in valence-arousal space)
share 2 of 3 neurons; `happy` and `sad` (distance 1.80) share none; overlap
tracks distance almost everywhere. That correlated representation didn't
cost accuracy — the full pipeline with these patterns lands within a point
of the disjoint-pattern version on every task (and actually beats it on
intent, tone, and plan). One real trap along the way, not smoothed over: the
first attempt used a bigger emotion group (25 neurons, for more room in the
similarity space) which quietly broke the *downstream* links — their
learning rate had been tuned against the old group's typical spike-count
scale, and a differently-sized group produces different-scale charge
(tone/plan/response accuracy collapsed to 43%/43%/7%). Same-size swap, same
link tuning, problem gone. The lesson generalizes: changing one cell
assembly's internal structure isn't local — everything it sends charge to
needs to be re-checked, not assumed fine.

**Novel probe:** less coherent than experiment 07's backprop version (which
produced "which one do you mean" then "restart the device now" for the same
two lines) but not obviously worse than experiment 08's associative-only
recall. Same underlying limit as experiment 08: this is pattern recall over
14 stored responses, not composition, so a genuinely novel input can only
ever land on whichever stored pattern is nearest — there's no way to
"almost" say the right thing.